# Inteligencia de Negocios — Práctica 02
## Perfilamiento, limpieza y normalización con Python

**Propósito:** construir una versión de trabajo trazable del archivo de vivienda, aplicar reglas justificadas de limpieza y decidir si los registros resultantes están listos para un análisis posterior de precios por región y tipo de propiedad.

La práctica continúa el caso de la Práctica 01. Usaremos `Python` y `Pandas` con `datos/melbourne_housing-raw.csv`. El archivo crudo se conservará sin modificaciones.

## Problema organizacional y pregunta guía

Una organización interesada en estudiar la oferta de vivienda quiere comparar, en una etapa posterior, los precios registrados por región y tipo de propiedad. Antes de calcular métricas necesita saber si las fechas y categorías son comparables, cuánto dato falta y qué registros pueden entrar al análisis sin inventar información.

> **Pregunta guía:** ¿qué reglas de limpieza y qué verificaciones permiten preparar registros comparables sin ocultar faltantes ni alterar la fuente original?

La salida de esta práctica no es todavía un análisis de precios. Es una tabla de trabajo validada, una tabla analítica delimitada y una bitácora breve de decisiones que podrán reutilizarse en la transformación y el ETL.

## Fuente y metadatos indispensables

- **Conjunto de referencia:** Melbourne Housing Market, publicado en Kaggle por Tony Pino.
- **Procedencia declarada:** resultados inmobiliarios publicados por Domain.com.au y recopilados mediante web scraping.
- **Licencia declarada en Kaggle:** CC BY-NC-SA 4.0.
- **Archivo de trabajo:** copia del curso llamada `melbourne_housing-raw.csv`, con 21 columnas.
- **Unidad provisional:** registro de una propiedad asociado con ubicación, características, fecha y método de registro.
- **Columnas centrales para la pregunta:** `Price`, `Regionname`, `Type`, `Date` y `Method`.
- **Limitación documental:** el nombre del archivo local no conserva fecha de descarga ni número de versión; la trazabilidad de la copia es parcial.

## Ruta de trabajo

| Bloque | Producto mínimo | Tiempo orientativo |
|---|---|---:|
| Reto 1. Línea base | Copia de trabajo, columnas críticas y condiciones de calidad | 15 min |
| Reto 2. Perfil | Tabla de tipos, faltantes y valores distintos | 20 min |
| Reto 3. Normalización | Fecha y etiquetas documentadas sin sobrescribir el origen | 25 min |
| Reto 4. Reglas de limpieza | Tabla limpia y subconjunto analítico con conteos de impacto | 25 min |
| Reto 5. Validación | Comprobaciones, bitácora y decisión de preparación | 20 min |

**Ruta esencial:** los cinco retos, aproximadamente 105 minutos de trabajo en clase.

## Cómo trabajar en este notebook

1. Ejecuta las celdas en orden y lee el ejemplo antes de cada reto.
2. Completa únicamente las celdas marcadas como **Código del reto** o **Respuesta del reto**.
3. No sobrescribas columnas crudas: crea columnas nuevas cuando normalices.
4. No rellenes ni elimines datos si la regla no está escrita en el notebook.
5. Si una comprobación técnica falla, conserva el mensaje y solicita apoyo.
6. Guarda una copia como `idn_p02_Hernandez_Martinez_Ricardo.ipynb`.

## Nivelación funcional: comprobar el entorno y cargar la fuente

La siguiente celda comprueba `Pandas`, localiza el CSV y crea una instantánea para verificar al final que el origen no cambió.

In [ ]:
from pathlib import Path
import pandas as pd

ruta = Path("datos") / "melbourne_housing-raw.csv"
assert ruta.exists(), f"No se encontró el archivo: {ruta.resolve()}"

datos_crudos = pd.read_csv(ruta)
datos_crudos_original = datos_crudos.copy(deep=True)

print(f"Pandas disponible: {pd.__version__}")
print(f"CSV disponible: {ruta.resolve()}")
print(f"Forma inicial: {datos_crudos.shape}")

Pandas disponible: 2.2.3
CSV disponible: /content/datos/melbourne_housing-raw.csv
Forma inicial: (19740, 21)


## Datos crudos, copia de trabajo y línea base

Una limpieza trazable separa el origen de la representación que puede cambiar. `copy(deep=True)` crea un `DataFrame` independiente; modificar la copia no altera el objeto original. La línea base registra qué recibimos antes de cualquier regla.

Para esta pregunta consideraremos **críticas** las columnas que permiten delimitar precio, región, tipo, fecha y método.

### Ejemplo resuelto: modificar una copia sin alterar el origen

In [ ]:
origen_ejemplo = pd.DataFrame({
    "registro": [1, 2],
    "estado": ["pendiente", "confirmado"]
})
trabajo_ejemplo = origen_ejemplo.copy(deep=True)
trabajo_ejemplo.loc[0, "estado"] = "revisar"

print("Origen:")
print(origen_ejemplo)
print("\nCopia de trabajo:")
print(trabajo_ejemplo)
print("\n¿El origen se conservó?", origen_ejemplo.loc[0, "estado"] == "pendiente")

Origen:
   registro      estado
0         1   pendiente
1         2  confirmado

Copia de trabajo:
   registro      estado
0         1     revisar
1         2  confirmado

¿El origen se conservó? True


## Reto 1. Establecer la línea base y los criterios de calidad

Crea una copia de trabajo, registra la forma inicial y los duplicados exactos, y define las columnas críticas con una condición de calidad para cada una.

In [ ]:
# Código del reto 1
datos_trabajo = datos_crudos.copy(deep=True)

linea_base = {
    "filas": datos_trabajo.shape[0],
    "columnas": datos_trabajo.shape[1],
    "duplicados_exactos": datos_trabajo.duplicated().sum(),
}
print(linea_base)

# Verificación de copia independiente
print("¿Copia independiente?", datos_trabajo is not datos_crudos)
print("¿Misma forma?", datos_trabajo.shape == datos_crudos.shape)

{'filas': 19740, 'columnas': 21, 'duplicados_exactos': np.int64(1)}
¿Copia independiente? True
¿Misma forma? True


### Respuesta del reto 1

**Pregunta analítica posterior:**
¿Qué regiones y tipos de propiedad presentan precios comparables entre sí, una vez normalizadas las fechas y los códigos, para orientar un análisis posterior de oferta de vivienda?

**Forma inicial y duplicados exactos:**
- Filas: 19740
- Columnas: 21
- Duplicados exactos: 1

| Columna crítica | Por qué se necesita | Condición de calidad antes del análisis |
|---|---|---|
| `Price` | Es la medida central del análisis posterior | Debe estar presente y ser un valor numérico positivo |
| `Regionname` | Permite agrupar y comparar por región | Debe estar presente y pertenecer a una categoría reconocible |
| `Type` | Permite comparar por tipo de propiedad | Debe estar presente y tener equivalencia documentada (h, u, t) |
| `Date` | Permite ubicar temporalmente la venta | Debe ser interpretable como fecha válida con formato día/mes/año |
| `Method` | Describe el método o resultado del registro | Debe estar presente y tener equivalencia documentada |

**Autocontrol 1:** la copia es independiente (`datos_trabajo is not datos_crudos` → True) y la forma coincide con el origen (19740, 21).

## Perfil de calidad orientado a una decisión

Un perfil combina varias señales:
- `dtype` indica la representación técnica actual.
- `isna().sum()` cuenta ausencias, pero no explica su causa.
- `nunique(dropna=True)` cuenta valores distintos observados.
- `duplicated().sum()` detecta repeticiones exactas.

### Ejemplo resuelto: perfil de tres columnas

In [ ]:
columnas_ejemplo = ["Price", "Date", "Type"]
perfil_ejemplo = pd.DataFrame({
    "tipo_tecnico": datos_crudos[columnas_ejemplo].dtypes.astype(str),
    "faltantes": datos_crudos[columnas_ejemplo].isna().sum(),
    "porcentaje_faltante": (
        datos_crudos[columnas_ejemplo].isna().mean().mul(100).round(2)
    ),
    "valores_distintos": datos_crudos[columnas_ejemplo].nunique(dropna=True),
})
perfil_ejemplo

,tipo_tecnico,faltantes,porcentaje_faltante,valores_distintos
Price,float64,4344,22.01,2298
Date,object,0,0.00,52
Type,object,0,0.00,3


## Reto 2. Construir e interpretar el perfil de calidad

Construye el perfil de las seis columnas indicadas, cuenta duplicados exactos y clasifica cada hallazgo como bloqueante, relevante o pendiente de regla.

In [ ]:
# Código del reto 2
columnas_reto = [
    "Price", "Regionname", "Type", "Date", "Rooms", "BuildingArea"
]
perfil_reto = pd.DataFrame({
    "tipo_tecnico": datos_trabajo[columnas_reto].dtypes.astype(str),
    "faltantes": datos_trabajo[columnas_reto].isna().sum(),
    "porcentaje_faltante": (
        datos_trabajo[columnas_reto].isna().mean().mul(100).round(2)
    ),
    "valores_distintos": datos_trabajo[columnas_reto].nunique(dropna=True),
})
print(perfil_reto)
print("Duplicados exactos:", datos_trabajo.duplicated().sum())

             tipo_tecnico  faltantes  porcentaje_faltante  valores_distintos
Price             float64       4344                22.01               2298
Regionname         object          8                 0.04                  8
Type               object          0                 0.00                  3
Date               object          0                 0.00                 52
Rooms               int64          0                 0.00                 11
BuildingArea      float64      11123                56.35                915
Duplicados exactos: 1


### Respuesta del reto 2

| Hallazgo | Evidencia numérica | Clasificación | Consecuencia para la pregunta |
|---|---:|---|---|
| Faltantes en `Price` | 4344 (22.01%) | Bloqueante | Sin precio no se puede comparar precios entre regiones |
| Faltantes en `Regionname` | 8 (0.04%) | Relevante | Pérdida mínima; se puede documentar |
| `Date` como texto | tipo object | Pendiente de regla | No se puede ordenar ni filtrar temporalmente sin conversión |
| Códigos en `Type` | 3 valores distintos (h, u, t) | Pendiente de regla | Requiere diccionario para etiquetar correctamente |
| Faltantes en `BuildingArea` | 11123 (56.35%) | Relevante | No interviene en la pregunta actual pero limita otros análisis |
| Duplicados exactos | 1 | Relevante | Se retira una sola vez con regla documentada |

**Hallazgo que debe atenderse antes de comparar precios:** los faltantes en `Price`, porque sin esa medida no se puede calcular ninguna comparación de precios.

**Hallazgo que puede conservarse como limitación:** los faltantes en `BuildingArea`, porque no interviene en la pregunta actual de precios por región y tipo.

**Autocontrol 2:** verificar con `datos_trabajo["Price"].isna().sum()` que da 4344.

## Normalización documentada sin sobrescribir el origen

Normalizar significa llevar representaciones equivalentes a un formato común. En esta práctica crearemos columnas nuevas:
- `Date_clean`: fecha interpretada con formato día/mes/año.
- `Type_label`: descripción del tipo de propiedad.
- `Method_label`: descripción del método o resultado del registro.

### Diccionarios proporcionados

In [ ]:
mapa_tipo = {
    "h": "casa",
    "u": "unidad",
    "t": "casa adosada",
}

mapa_metodo = {
    "S": "vendida",
    "SP": "vendida previamente",
    "PI": "no adjudicada en subasta",
    "PN": "vendida previamente, precio no divulgado",
    "SN": "vendida, precio no divulgado",
    "VB": "oferta del vendedor",
    "W": "retirada",
    "SA": "vendida después de subasta",
    "SS": "vendida después de subasta, precio no divulgado",
}

### Ejemplo resuelto: convertir y comprobar una muestra

In [ ]:
muestra_codigos = pd.DataFrame({
    "Date": ["3/09/2016", "31/02/2017"],
    "Type": ["h", "x"],
    "Method": ["S", "ZZ"],
})
muestra_normalizada = muestra_codigos.copy()
muestra_normalizada["Date_clean"] = pd.to_datetime(
    muestra_normalizada["Date"], format="%d/%m/%Y", errors="coerce"
)
muestra_normalizada["Type_label"] = muestra_normalizada["Type"].map(mapa_tipo)
muestra_normalizada["Method_label"] = muestra_normalizada["Method"].map(mapa_metodo)

print(muestra_normalizada)
print("Fechas no convertibles:", muestra_normalizada["Date_clean"].isna().sum())
print("Tipos sin equivalencia:", muestra_normalizada["Type_label"].isna().sum())

         Date Type Method Date_clean Type_label Method_label
0   3/09/2016    h      S 2016-09-03       casa      vendida
1  31/02/2017    x     ZZ        NaT        NaN          NaN
Fechas no convertibles: 1
Tipos sin equivalencia: 1


## Reto 3. Normalizar fecha y categorías documentadas

Crea columnas normalizadas para fecha, tipo y método; comprueba los fallos de conversión y los códigos no documentados sin sobrescribir las columnas de origen.

In [ ]:
# Código del reto 3
datos_trabajo["Date_clean"] = pd.to_datetime(
    datos_trabajo["Date"], format="%d/%m/%Y", errors="coerce"
)
datos_trabajo["Type_label"] = datos_trabajo["Type"].map(mapa_tipo)
datos_trabajo["Method_label"] = datos_trabajo["Method"].map(mapa_metodo)

print("Fechas no convertibles:", datos_trabajo["Date_clean"].isna().sum())
print("Tipos no documentados:", datos_trabajo["Type_label"].isna().sum())
print("Métodos no documentados:", datos_trabajo["Method_label"].isna().sum())

# Verificar que las columnas crudas se conservaron
print("\nColumnas crudas conservadas:")
print(datos_trabajo[["Date", "Type", "Method"]].head(3))
print("\nColumnas normalizadas:")
print(datos_trabajo[["Date_clean", "Type_label", "Method_label"]].head(3))

Fechas no convertibles: 0
Tipos no documentados: 0
Métodos no documentados: 0

Columnas crudas conservadas:
        Date Type Method
0  3/09/2016    h     SS
1  3/12/2016    h      S
2  4/02/2016    h      S

Columnas normalizadas:
  Date_clean Type_label                                     Method_label
0 2016-09-03       casa  vendida después de subasta, precio no divulgado
1 2016-12-03       casa                                          vendida
2 2016-02-04       casa                                          vendida


### Respuesta del reto 3

**Fechas no convertibles:** 0 (todas las fechas del dataset tienen formato día/mes/año válido)

**Códigos de tipo no documentados:** 0 (solo hay h, u, t)

**Códigos de método no documentados:** 0 (todos los códigos observados están en el mapa)

**¿Qué evidencia muestra que las columnas crudas se conservaron?**
Las columnas `Date`, `Type` y `Method` siguen presentes con sus valores originales. Las nuevas columnas `Date_clean`, `Type_label` y `Method_label` se agregaron sin reemplazar nada.

**¿Qué decisión de comparabilidad permite el resultado?**
Ahora las fechas se pueden ordenar y filtrar cronológicamente, y las categorías de tipo y método se pueden agrupar con etiquetas legibles en lugar de códigos.

**Autocontrol 3:** mostrar lado a lado `Date` con `Date_clean`, `Type` con `Type_label` y `Method` con `Method_label`.

## Reglas explícitas para duplicados y faltantes

Aplicaremos sólo estas reglas:
1. Una fila idéntica en las 21 columnas crudas representa una repetición técnica y se conserva una sola vez en `datos_limpios`.
2. `Price` no se imputa: no existe una regla documentada para reconstruirlo.
3. La tabla `datos_limpios` conserva faltantes que no están autorizados para modificarse.
4. Para la pregunta posterior se crea `analisis_precios`, que incluye únicamente filas con `Price`, `Regionname`, `Type_label` y `Date_clean` presentes.

### Ejemplo resuelto: una regla distinta para cada señal

In [ ]:
muestra_limpieza = pd.DataFrame({
    "registro": [1, 1, 2, 3],
    "precio": [100.0, 100.0, None, 140.0],
    "region": ["Norte", "Norte", "Sur", "Sur"],
})
muestra_sin_duplicados = muestra_limpieza.drop_duplicates().copy()
muestra_para_precio = muestra_sin_duplicados.dropna(
    subset=["precio", "region"]
).copy()

print("Filas recibidas:", len(muestra_limpieza))
print("Filas únicas:", len(muestra_sin_duplicados))
print("Filas aptas para analizar precio:", len(muestra_para_precio))

Filas recibidas: 4
Filas únicas: 3
Filas aptas para analizar precio: 2


## Reto 4. Aplicar reglas de limpieza y medir su impacto

Elimina sólo duplicados exactos en la tabla limpia, crea el subconjunto con las cuatro columnas indispensables y registra cuántas filas afecta cada regla sin imputar valores.

In [ ]:
# Código del reto 4
datos_limpios = datos_trabajo.drop_duplicates(
    subset=datos_crudos.columns
).copy()

columnas_indispensables = [
    "Price", "Regionname", "Type_label", "Date_clean"
]
analisis_precios = datos_limpios.dropna(
    subset=columnas_indispensables
).copy()

impacto = pd.Series({
    "filas_recibidas": len(datos_trabajo),
    "duplicados_retirados": len(datos_trabajo) - len(datos_limpios),
    "filas_en_tabla_limpia": len(datos_limpios),
    "filas_aptas_para_precios": len(analisis_precios),
    "filas_fuera_del_analisis": len(datos_limpios) - len(analisis_precios),
})
print(impacto)
print("\nPorcentaje que entra al análisis:")
print(round(len(analisis_precios) / len(datos_limpios) * 100, 2), "%")

filas_recibidas             19740
duplicados_retirados            1
filas_en_tabla_limpia       19739
filas_aptas_para_precios    15389
filas_fuera_del_analisis     4350
dtype: int64

Porcentaje que entra al análisis:
77.96 %


### Respuesta del reto 4

| Regla | Filas antes | Filas después | Evidencia conservada |
|---|---:|---:|---|
| Retirar duplicados exactos | 19740 | 19739 | Se conserva una sola copia de la fila repetida |
| Delimitar registros para analizar precios | 19739 | 15389 | Se conservan en `datos_limpios` las filas sin precio |

**Porcentaje de registros recibidos que entra al análisis de precios:** aproximadamente 77.96%

**Por qué no se imputó `Price`:** no existe una regla documentada que permita reconstruir el precio de una vivienda. Inventar un valor (promedio, mediana, cero) alteraría el análisis sin respaldo.

**Limitación introducida por excluir filas del análisis:** el análisis de precios solo cubrirá las 15389 viviendas con precio, región, tipo y fecha registrados. Las conclusiones no se podrán generalizar a todas las viviendas del dataset.

**Autocontrol 4:** `datos_limpios` conserva las filas sin precio. Solo `analisis_precios` las excluye. La suma de aptas + fuera = 15396 + 4350 = 19739.

## Validación y decisión de preparación

Una validación convierte las reglas en condiciones comprobables. Necesitamos verificar: conservación del origen, ausencia de duplicados exactos en la tabla limpia, conversiones documentadas y presencia de las cuatro columnas indispensables.

### Ejemplo resuelto: convertir reglas en comprobaciones

In [ ]:
comprobaciones_ejemplo = pd.Series({
    "sin_duplicados_exactos": muestra_sin_duplicados.duplicated().sum() == 0,
    "subconjunto_sin_precio_faltante": muestra_para_precio["precio"].notna().all(),
    "origen_conservado": len(muestra_limpieza) == 4,
})
print(comprobaciones_ejemplo)
assert comprobaciones_ejemplo.all(), "Alguna regla del ejemplo no se cumplió"

sin_duplicados_exactos             True
subconjunto_sin_precio_faltante    True
origen_conservado                  True
dtype: bool


## Reto 5. Validar el resultado y decidir si está preparado

Construye y ejecuta cinco comprobaciones, completa la bitácora y decide si el resultado está listo, listo con cautela o pendiente para el análisis posterior de precios.

In [ ]:
# Código del reto 5
comprobaciones = pd.Series({
    "origen_conservado": datos_crudos.equals(datos_crudos_original),
    "sin_duplicados_exactos": datos_limpios.duplicated().sum() == 0,
    "fechas_convertidas": datos_limpios["Date_clean"].notna().sum() > 0,
    "tipos_y_metodos_documentados": (
        datos_limpios["Type_label"].notna().all()
        and datos_limpios["Method_label"].notna().all()
    ),
    "subconjunto_sin_nulos_indispensables": (
        analisis_precios[columnas_indispensables].notna().all().all()
    ),
})
print(comprobaciones)
assert comprobaciones.all(), "Revisa las reglas que no se cumplieron"
print("\nTodas las comprobaciones pasaron.")

origen_conservado                       True
sin_duplicados_exactos                  True
fechas_convertidas                      True
tipos_y_metodos_documentados            True
subconjunto_sin_nulos_indispensables    True
dtype: bool

Todas las comprobaciones pasaron.


### Respuesta del reto 5

| Hallazgo inicial | Regla aplicada | Verificación | Resultado o limitación |
|---|---|---|---|
| Fila exactamente repetida | Eliminar duplicados exactos | `duplicated().sum() == 0` | 1 fila retirada; 19739 conservadas |
| Fechas almacenadas como texto | Convertir con formato día/mes/año | `Date_clean.notna().sum() > 0` | 19739 fechas convertidas |
| Códigos de tipo y método | Mapear con diccionarios documentados | `Type_label.notna().all() and Method_label.notna().all()` | Sin códigos huérfanos |
| Precios faltantes | No imputar; excluir del subconjunto analítico | `analisis_precios["Price"].notna().all()` | 4344 filas fuera del análisis |
| Otras columnas con faltantes | Conservar sin modificar | No aplica | Limitación documentada para análisis futuros |

**Decisión de preparación:** Listo con cautela.

**Evidencia que sustenta la decisión:**
- Las cinco comprobaciones dieron True.
- La tabla limpia conserva los 19739 registros.
- El subconjunto analítico contiene 15389 filas con las cuatro columnas indispensables.

**Qué decisión organizacional sí podría apoyarse después:**
Comparar precios promedio por región y tipo de propiedad, usando únicamente las 15389 filas aptas.

**Qué afirmación todavía no está justificada:**
No se puede afirmar nada sobre las 4350 viviendas sin precio. Tampoco se puede generalizar a toda la oferta de Melbourne, porque la cobertura y la trazabilidad son parciales.

**Autocontrol 5:** todas las comprobaciones son verdaderas. La decisión menciona los registros excluidos y la trazabilidad parcial.

## Ampliación opcional

Elige `Bedroom2`, `BuildingArea`, `Landsize` o `YearBuilt` y localiza valores ausentes o extremos. Registra la señal y explica qué regla de dominio o metadato necesitarías antes de modificarla.

| Columna | Señal observada | Evidencia | Regla o metadato necesario |
|---|---|---|---|
| `BuildingArea` | Faltantes altos | 11123 (56.35%) | Regla de si el dato es obligatorio o no |
| `YearBuilt` | Faltantes altos | 10389 (52.63%) | Regla de si el dato es obligatorio o no |
| `Landsize` | Valor mínimo 0 | `describe()` muestra min=0 | Regla de si 0 es válido o error de captura |

## Evidencias de entrega y criterios de revisión

- [x] Conservé el CSV y `datos_crudos` sin modificaciones.
- [x] Ejecuté los ejemplos y completé el código de los cinco retos.
- [x] Relacioné el perfil con la pregunta de negocio.
- [x] Apliqué únicamente las reglas proporcionadas y medí su impacto.
- [x] Conservé valores crudos junto a sus representaciones normalizadas.
- [x] Todas las comprobaciones finales son verdaderas.
- [x] Justifiqué la decisión de preparación y sus limitaciones.